# 05 Find the source — notebook

Chapter 04 showed *when*, *where* and *who*. Now Maya tests **why**: which exposures
after Typhoon Amihan made leptospirosis more likely? Doc Mon's team interviewed
**120 cases and 240 controls** (two neighbours of the same barangay for each case) in
the eight riverside barangays. The study, the people and every number are synthetic.

You will build 2×2 tables with `pd.crosstab`, compute odds ratios and 95% confidence
intervals by hand, check them against scipy and statsmodels, catch a flipped table, and
untangle a confounder with the Mantel–Haenszel method.

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

## Step 1 · Load the study and look

Why a **case-control study**? Leptospirosis is uncommon even in a flood, so following
the whole city (a *cohort*) would mean interviewing tens of thousands of people to find
a few hundred cases. Instead we start from the **cases** we already have, pick
**controls** who did not get sick from the same barangays, and compare what both groups
did after the typhoon.

In [ ]:
import math

import matplotlib.pyplot as plt
import pandas as pd
import statsmodels.api as sm
from scipy.stats.contingency import odds_ratio

cc = pd.read_csv("data/synthetic/ph/lepto_case_control.csv")
print(len(cc), "participants:", cc["status"].value_counts().to_dict())
print("Matched sets:", cc["matched_set"].nunique(), "| barangays:", ", ".join(sorted(cc["barangay"].unique())))
cc.head(6)

## Step 2 · Tala got it wrong: the upside-down table

> **Ask Tala:** "In `cc`, `waded_floodwater` is 0/1 and `status` is `case`/`control`.
> Make the 2×2 table and compute the odds ratio for wading."

Tala builds the table and reads the four cells **by position**: top-left is *a*,
top-right *b*, and so on.

In [ ]:
# Tala's first answer -- cells read by POSITION, without looking at the labels
t = pd.crosstab(cc["waded_floodwater"], cc["status"])
a, b = t.iloc[0, 0], t.iloc[0, 1]
c, d = t.iloc[1, 0], t.iloc[1, 1]
print(t)
print("Tala's OR =", round(a * d / (b * c), 2))

An OR of 0.18 would mean wading **protected** people. Aling Nena would laugh. How do we
catch it without knowing the answer?

1. **Read the labels.** `crosstab` sorts the rows 0 before 1, so the *first* row is
   "did **not** wade". Tala's *a* is unexposed cases, not exposed cases. The formula
   assumed one order; the table had the other.
2. **Sanity-check the direction.** Before any formula, compare simple percentages: did
   more cases than controls wade? If yes, the OR must be above 1.

(Swapping rows *and* columns — a full transpose — would not change the OR. Flipping
only one of them gives the **reciprocal**: 0.18 = 1 / 5.57.)

In [ ]:
waded_pct = pd.crosstab(cc["status"], cc["waded_floodwater"], normalize="index") * 100
print("Percent who waded:")
print(waded_pct[1].round(1))

Three in four cases waded, against about one in three controls: the OR must be well
above 1. Now the safe way. We put the rows and columns in a **fixed, labelled order**
with `.loc[[1, 0], [1, 0]]` and name them, so anyone can read the table:

|  | Cases | Controls |
|---|---|---|
| **Exposed** | a | b |
| **Not exposed** | c | d |

$$OR = \frac{a \times d}{b \times c} \qquad
95\%\ CI = \exp\left(\ln OR \pm 1.96\sqrt{\tfrac1a+\tfrac1b+\tfrac1c+\tfrac1d}\right)$$

This confidence interval is the *Woolf* (log) method.

In [ ]:
def two_by_two(data: pd.DataFrame, exposure: str) -> pd.DataFrame:
    """2x2 table: rows exposed / not exposed, columns cases / controls."""
    table = pd.crosstab(data[exposure], data["case"]).loc[[1, 0], [1, 0]]
    table.index = ["Exposed", "Not exposed"]
    table.columns = ["Cases", "Controls"]
    return table


table = two_by_two(cc, "waded_floodwater")
(a, b), (c, d) = table.to_numpy()
or_wading = a * d / (b * c)
se = math.sqrt(1 / a + 1 / b + 1 / c + 1 / d)
low, high = math.exp(math.log(or_wading) - 1.96 * se), math.exp(math.log(or_wading) + 1.96 * se)

print(table, "\n")
print(f"a={a} b={b} c={c} d={d}")
print(f"OR = {or_wading:.2f} (95% CI {low:.2f}–{high:.2f})")

**Check it** with two independent libraries. scipy's `odds_ratio` must be told
`kind="sample"` (its default is a different estimator, the *conditional* OR, which gives
a slightly different number). statsmodels' `Table2x2` uses the same Woolf interval.

In [ ]:
scipy_result = odds_ratio(table.to_numpy(), kind="sample")
scipy_ci = scipy_result.confidence_interval(confidence_level=0.95)
sm_table = sm.stats.Table2x2(table.to_numpy())
sm_low, sm_high = sm_table.oddsratio_confint()

print(f"by hand     : {or_wading:.2f} ({low:.2f}–{high:.2f})")
print(f"scipy       : {scipy_result.statistic:.2f} ({scipy_ci.low:.2f}–{scipy_ci.high:.2f})")
print(f"statsmodels : {sm_table.oddsratio:.2f} ({sm_low:.2f}–{sm_high:.2f})")
print(f"scipy default (conditional): {odds_ratio(table.to_numpy()).statistic:.2f}")

assert math.isclose(scipy_result.statistic, or_wading) and math.isclose(sm_table.oddsratio, or_wading)
assert math.isclose(scipy_ci.low, low, rel_tol=1e-3) and math.isclose(sm_high, high, rel_tol=1e-3)

## Step 3 · Risk ratio or odds ratio?

A **risk ratio** (RR) compares the *risk* of disease in the exposed with the risk in the
unexposed: $\frac{a/(a+b)}{c/(c+d)}$. That needs everyone at risk, as in a cohort. In a
case-control study **we chose** how many controls to take, so "risk" in our table is
made up by our design. Watch what happens if we had taken **one** control per case
instead of two: the "RR" moves, the OR barely does.

In [ ]:
def fake_rr(t: pd.DataFrame) -> float:
    (a, b), (c, d) = t.to_numpy()
    return (a / (a + b)) / (c / (c + d))


one_control = cc[(cc["case"] == 1) | ~cc.duplicated(["matched_set", "case"])]  # first control only
for label, data in [("2 controls per case", cc), ("1 control per case", one_control)]:
    t = two_by_two(data, "waded_floodwater")
    print(f"{label}: n = {len(data)} | OR = {sm.stats.Table2x2(t.to_numpy()).oddsratio:.2f}"
          f" | 'RR' = {fake_rr(t):.2f}  <- not a real risk ratio")

So: **cohort → RR; case-control → OR.** When a disease is rare in the population, the OR
is close to the RR, which is why we can still talk about "about five times the risk"
with care.

## Step 4 · Boots among waders, and doxycycline

Boots only matter for people who **were** in the water, so the boots question was asked
only of waders (`wore_boots` is blank for everyone else). We restrict to waders first.
Doxycycline prophylaxis was given at evacuation centres and to clean-up volunteers.
An OR **below 1** means the exposure was less common among cases: a protective factor.

In [ ]:
waders = cc[cc["waded_floodwater"] == 1]
for label, data, exposure in [("Wore boots (waders only)", waders, "wore_boots"),
                              ("Doxycycline prophylaxis", cc, "doxy_prophylaxis")]:
    t = two_by_two(data, exposure)
    result = sm.stats.Table2x2(t.to_numpy())
    lo, hi = result.oddsratio_confint()
    print(f"{label}: n = {len(data)}")
    print(t)
    print(f"OR = {result.oddsratio:.2f} (95% CI {lo:.2f}–{hi:.2f})\n")

## Step 5 · All exposures in one picture

> **Ask Tala:** "Using my function `two_by_two(data, exposure)` (rows exposed/not
> exposed, columns cases/controls), build a table with a, b, c, d, OR and 95% CI for
> several 0/1 columns, then draw a forest plot with a log x-axis and a line at OR = 1."

`rats_near_home` is a useful *null* exposure: rats matter for leptospirosis in general,
but in this study seeing rats near the house did not distinguish cases from controls.

In [ ]:
cc["male"] = (cc["sex"] == "M").astype(int)
EXPOSURES = [("waded_floodwater", cc), ("wore_boots", waders), ("doxy_prophylaxis", cc),
             ("skin_wound", cc), ("cleanup_work", cc), ("male", cc), ("rats_near_home", cc)]

rows = []
for exposure, data in EXPOSURES:
    t = two_by_two(data, exposure)
    result = sm.stats.Table2x2(t.to_numpy())
    lo, hi = result.oddsratio_confint()
    (a, b), (c, d) = t.to_numpy()
    label = exposure + (" (waders only)" if data is waders else "")
    rows.append({"exposure": label, "a": a, "b": b, "c": c, "d": d,
                 "OR": round(result.oddsratio, 2), "CI low": round(lo, 2), "CI high": round(hi, 2)})
summary = pd.DataFrame(rows).set_index("exposure")

fig, ax = plt.subplots(figsize=(7, 4))
y = list(range(len(summary)))[::-1]  # first exposure at the top
for yi, (name, row) in zip(y, summary.iterrows()):
    # red = risk factor, green = protective, grey = CI crosses 1
    colour = "#CE1126" if row["CI low"] > 1 else "#2F7D32" if row["CI high"] < 1 else "#6E7882"
    ax.plot([row["CI low"], row["CI high"]], [yi, yi], color=colour, linewidth=2.5)
    ax.plot(row["OR"], yi, "o", color=colour, markersize=8)
ax.axvline(1, color="black", linestyle="--", linewidth=1)
ax.set_xscale("log")
ax.set_xticks([0.2, 0.5, 1, 2, 5, 10], labels=["0.2", "0.5", "1", "2", "5", "10"])
ax.set_yticks(y, labels=summary.index)
ax.set_xlabel("Odds ratio (95% CI, log scale)")
ax.set_title("Exposures and leptospirosis, crude ORs (synthetic study)")
plt.tight_layout()
plt.show()
summary

## Step 6 · Confounding: was it the wading, or the clean-up?

Doc Mon looks at the table and asks: *"Clean-up volunteers waded more, didn't they? And
clean-up work had its own risks — wounds, long hours in mud. Could part of the wading OR
belong to clean-up?"*

A **confounder** is a third factor that is linked to the exposure *and* is a risk
factor for the disease on its own, so it mixes its effect into the crude OR. Check the
first link among the **controls** (they show us how exposures go together in the
population).

In [ ]:
controls = cc[cc["case"] == 0]
print("Controls who waded, by clean-up work (%):")
print((pd.crosstab(controls["cleanup_work"], controls["waded_floodwater"], normalize="index")[1] * 100).round(1))
print()
print("Crude OR for clean-up work itself:", summary.loc["cleanup_work", "OR"])

Clean-up workers waded much more often, and clean-up work is itself linked to the
disease. So we **stratify**: compute the wading OR separately among people who did
clean-up work and people who did not. Then we combine the two strata with the
**Mantel–Haenszel** (MH) odds ratio, a weighted average of the stratum ORs:

$$OR_{MH} = \frac{\sum_i a_i d_i / n_i}{\sum_i b_i c_i / n_i}$$

where *i* is the stratum and *n_i* the number of people in it.

In [ ]:
numerator = denominator = 0.0
strata = []
for work, group in cc.groupby("cleanup_work"):
    t = two_by_two(group, "waded_floodwater")
    (a, b), (c, d) = t.to_numpy()
    n = a + b + c + d
    numerator += a * d / n
    denominator += b * c / n
    strata.append(t.to_numpy())
    label = "did clean-up work" if work == 1 else "no clean-up work"
    print(f"{label:<18} a={a:>2} b={b:>2} c={c:>2} d={d:>3}  stratum OR = {a * d / (b * c):.2f}")

or_mh = numerator / denominator
print(f"\nCrude OR = {or_wading:.2f}   Mantel-Haenszel OR = {or_mh:.2f}")

**Check it** with statsmodels' `StratifiedTable`, which also gives the MH confidence
interval and a test of whether the stratum ORs differ more than chance would explain.

In [ ]:
st = sm.stats.StratifiedTable(strata)
mh_low, mh_high = st.oddsratio_pooled_confint()
print(f"statsmodels MH OR = {st.oddsratio_pooled:.2f} (95% CI {mh_low:.2f}–{mh_high:.2f})")
print(f"Test of equal stratum ORs (Breslow-Day): p = {st.test_equal_odds().pvalue:.2f}")
assert math.isclose(st.oddsratio_pooled, or_mh)
change = (or_wading - or_mh) / or_wading * 100
print(f"Adjusting for clean-up work moved the OR from {or_wading:.2f} to {or_mh:.2f} ({change:.0f}% smaller)")

**What changed and why.** The crude OR (5.57) mixed two things: the risk of wading
itself and the extra risk of clean-up work, because clean-up workers waded more. Within
each stratum the clean-up factor is held fixed, and the combined MH OR falls to **4.28**.
Wading is still strongly linked to leptospirosis — the association did not disappear —
but part of the crude number belonged to clean-up work. The stratum ORs (5.56 and 2.88)
look different, but with these numbers the test cannot tell them apart from chance
(p ≈ 0.24), so one MH summary is reasonable. Real studies have many tangled exposures
and sampling error, so crude ORs are rarely the whole truth.

**What Maya recommends to the City Health Officer:**

> Wading in floodwater after Amihan was strongly associated with leptospirosis, even
> after accounting for clean-up work (adjusted OR about 4), and wearing boots was
> associated with much lower odds among those who waded. Before the next flood, the city
> should supply boots and wound care to clean-up crews and residents who must wade, and
> tell anyone with fever after floodwater exposure to consult early. Doxycycline
> prophylaxis was associated with lower odds of disease in this study, so its
> distribution to high-exposure groups should continue, following current DOH guidance.

## Your turn

1. **Skin wounds.** Is the `skin_wound` OR confounded by clean-up work too? *Hint: copy
   the Mantel–Haenszel cell and change `"waded_floodwater"` to `"skin_wound"`.*
2. **Are men really at higher risk?** The crude OR for `male` is 2.82. Stratify it by
   `waded_floodwater` and compute the MH OR. *Hint: men waded more often; expect the OR
   to shrink.*
3. **Reproduce Tala's mistake on purpose.** Flip only the rows of the correct table with
   `table.iloc[::-1]` and compute $ad/bc$. Show that the answer is `1 / or_wading`.
   *Hint: `math.isclose(...)` is your friend.*

In [ ]:
# Your turn: try the exercises here